# AIC26 Modal Florence-2 Image Captioning — Kaggle Runner

This notebook runs **Florence-2 Image Captioning** on **Modal Serverless GPU (T4)** from Kaggle.

**Configuration:**
- Task: Batch 3 of 3 (`--batch-index 2 --num-batches 3`)
- Input: Keyframes dataset attached under `/kaggle/input/...`
- Output: `/kaggle/working/captioning_batch3`

**Prerequisites:** Add `MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET` in Kaggle **Add-ons -> Secrets** or fill in Cell 2.

In [ ]:
# --- 1. CLONE REPOSITORY & INSTALL MODAL SDK ---
import os, sys, shutil

GIT_REPO_URL = "https://github.com/Hoaiduc195/aic2026.git"
GIT_BRANCH = "main"
REPO_DIR = "/kaggle/working/aic2026"

shutil.rmtree(REPO_DIR, ignore_errors=True)
print(f"Cloning repo from GitHub: {GIT_REPO_URL} (branch: {GIT_BRANCH})")
!git clone --depth 1 -b {GIT_BRANCH} {GIT_REPO_URL} {REPO_DIR}

print("Installing Modal SDK...")
!pip install -q modal

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repository cloned and Modal SDK installed!")

In [ ]:
# --- 2. MODAL AUTHENTICATION TOKENS ---
# Get token ID and Secret from https://modal.com/settings/tokens
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["MODAL_TOKEN_ID"] = user_secrets.get_secret("MODAL_TOKEN_ID")
    os.environ["MODAL_TOKEN_SECRET"] = user_secrets.get_secret("MODAL_TOKEN_SECRET")
    print("Loaded Modal credentials from Kaggle Secrets!")
except Exception:
    # Fallback: Paste your tokens here directly if not using Kaggle Secrets
    os.environ["MODAL_TOKEN_ID"] = "ak-xxxxxx"        # <--- Paste your Modal Token ID here
    os.environ["MODAL_TOKEN_SECRET"] = "as-xxxxxx"    # <--- Paste your Modal Token Secret here
    print("Using direct Modal environment tokens.")

In [ ]:
# --- 3. CONFIGURATION & DISCOVER INPUT PATH ---
import glob

CANDIDATE_PATHS = [
    "/kaggle/input/deduped-keyframes/deduped_keyframes",
    "/kaggle/input/datasets/hoicdng/aic-2026-keyframes/keyframes",
    "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/map-keyframes",
    "/kaggle/input/btc-keyframes",
    "/kaggle/input/keyframes"
]

INPUT_DIR = None
for cand in CANDIDATE_PATHS:
    if os.path.exists(cand) and glob.glob(f"{cand}/*"):
        INPUT_DIR = cand
        break

if not INPUT_DIR:
    hits = glob.glob("/kaggle/input/**/keyframes", recursive=True)
    INPUT_DIR = hits[0] if hits else "/kaggle/input"

OUTPUT_DIR = "/kaggle/working/captioning_batch3"
BATCH_INDEX = 2  # Batch 3 of 3 (0-indexed: Batch 1=0, Batch 2=1, Batch 3=2)
NUM_BATCHES = 3
BUDGET_USD = 25.0

print(f"INPUT_DIR   : {INPUT_DIR}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"BATCH_INDEX : {BATCH_INDEX} (Batch 3/3)")

In [ ]:
# --- 4. EXECUTE MODAL FLORENCE-2 CAPTIONING (BATCH 3) ---
SCRIPT_PATH = os.path.join(REPO_DIR, "pipelines", "feature_extraction", "captioning", "modal_florence_captioning.py")

!modal run {SCRIPT_PATH} \
  --input-dir "{INPUT_DIR}" \
  --output-dir "{OUTPUT_DIR}" \
  --batch-index {BATCH_INDEX} \
  --num-batches {NUM_BATCHES} \
  --budget-usd {BUDGET_USD}

In [ ]:
# --- 5. COMPRESS RESULTS TO ZIP FOR DOWNLOAD ---
import zipfile

ZIP_PATH = "/kaggle/working/captioning_batch3.zip"
if os.path.exists(OUTPUT_DIR):
    print(f"Zipping captioning results from {OUTPUT_DIR} -> {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for f in files:
                full_p = os.path.join(root, f)
                rel_p = os.path.relpath(full_p, OUTPUT_DIR)
                z.write(full_p, rel_p)
    size_mb = os.path.getsize(ZIP_PATH) / 2**20
    print(f"Successfully created {ZIP_PATH} ({size_mb:.2f} MB)!")
else:
    print("OUTPUT_DIR does not exist yet. Make sure step 4 completed.")